# AllTrails Export Control Notebook

## Contol

In [4]:
# Control Cell
TEST_TRAIL_URL_EXPORT = False
TEST_TRAIL_GEOMETRY_EXPORT = False
TEST_ACTIVITY_URL_EXPORT = False
TEST_ACTIVITY_EXPORT = False

AUTO_ACTIVITY_URL_EXPORT = True
AUTO_ACTIVITY_EXPORT = True

## Setup

In [6]:
# pip install playwright
# python -m playwright install chromium

import random
import subprocess
import sys
import time
from pathlib import Path
import pandas as pd
import numpy as np

# Random pause range (minutes) between automated export calls
MIN_DELAY_MINUTES = 1
MAX_DELAY_MINUTES = 2

In [11]:
REGION_URL = "https://www.alltrails.com/parks/new-zealand/canterbury/banks-peninsula"
TRAIL_URL = "https://www.alltrails.com/trail/new-zealand/canterbury/onawe-pa-track"
ACTIVITY_URL = "https://www.alltrails.com/explore/recording/le-bons-bay-hiking-5f268df"
OUT_DIR = Path("Data\\AllTrails\\Downloads")

# Reuse a persistent browser profile so login/challenge can be solved once and reused
# also export storage state after successful access
PROFILE_DIR = Path("Data\\AllTrails\\alltrails_profile")
STORAGE_STATE_FILE = Path("Data\\AllTrails\\storage_state.json")

# URL export writes the shared activity index
INDEX_CSV = OUT_DIR / "activity_index.csv"

# Set this env var in your shell to keep anonymized IDs salted
ID_SALT_ENV = "ALLTRAILS_ID_SALT"

## Testing Calls

In [4]:
if TEST_TRAIL_URL_EXPORT:
    try:
        result = subprocess.run(
            [
                sys.executable,
                "alltrails_trail_urls_export.py",
                "--url",
                REGION_URL,
                "--out-dir",
                str(OUT_DIR),
                "--profile-dir",
                str(PROFILE_DIR),
                "--storage-state",
                str(STORAGE_STATE_FILE),
            ],
            check=True,
            cwd=str(Path.cwd()),
            capture_output=True,
            text=True,
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
    except subprocess.CalledProcessError as exc:
        if exc.stdout:
            print(exc.stdout)
        if exc.stderr:
            print(exc.stderr)
        raise

In [5]:
if TEST_TRAIL_GEOMETRY_EXPORT:
    # Download GPX files for each trail listed in the Banks Peninsula track URL CSV
    subprocess.run(
        [
            sys.executable,
            "alltrails_trail_data_export.py",
            "--index-csv",
            "Data/AllTrails_Downloads/banks-peninsula_track_urls.csv",
            "--out-dir",
            str(OUT_DIR),
            "--profile-dir",
            str(PROFILE_DIR),
            "--storage-state",
            str(STORAGE_STATE_FILE),
            "--start-row",
            "30",
            "--max-items",
            "35",
            "--retries",
            "1",
            "--menu-timeout-ms",
            "30000",
            "--download-timeout-ms",
            "25000",
            "--slow-mo-ms",
            "5000",
        ],
        check=True,
        cwd=str(Path.cwd()),
    )

In [6]:
if TEST_ACTIVITY_URL_EXPORT:
    try:
        result = subprocess.run(
            [
                sys.executable,
                "alltrails_activity_urls_export.py",
                "--url",
                TRAIL_URL,
                "--out-dir",
                str(OUT_DIR),
                "--profile-dir",
                str(PROFILE_DIR),
                "--storage-state",
                str(STORAGE_STATE_FILE),
                "--index-csv",
                str(INDEX_CSV),
                "--id-salt-env",
                ID_SALT_ENV,
            ],
            check=True,
            cwd=str(Path.cwd()),
            capture_output=True,
            text=True,
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
    except subprocess.CalledProcessError as exc:
        if exc.stdout:
            print(exc.stdout)
        if exc.stderr:
            print(exc.stderr)
        raise

In [7]:
if TEST_ACTIVITY_EXPORT:
    # Quick sanity run: download 1 activity from the shared index CSV
    subprocess.run(
        [
            sys.executable,
            "alltrails_activity_data_export.py",
            "--index-csv",
            str(INDEX_CSV),
            "--out-dir",
            str(OUT_DIR),
            "--format",
            "csv",
            "--profile-dir",
            str(PROFILE_DIR),
            "--storage-state",
            str(STORAGE_STATE_FILE),
            "--start-row",
            "1",
            "--max-items",
            "1",
            "--retries",
            "1",
            "--menu-timeout-ms",
            "5000",
            "--download-timeout-ms",
            "16000",
            "--slow-mo-ms",
            "0",
        ],
        check=True,
        cwd=str(Path.cwd()),
    )

## Auto Extraction

In [ ]:
from datetime import datetime
import csv

TRACK_URLS_CSV = Path("Data/AllTrails/Downloads/banks-peninsula_track_urls.csv")

if TRACK_URLS_CSV.exists():
    banks_peninsula_trail_urls = pd.read_csv(TRACK_URLS_CSV)
    banks_peninsula_trail_urls["Extracted"] = False
    banks_peninsula_trail_urls.to_csv(TRACK_URLS_CSV, index=False)
    print(f"Reset Extracted=False for {len(banks_peninsula_trail_urls)} tracks in {TRACK_URLS_CSV}")
else:
    raise FileNotFoundError(TRACK_URLS_CSV)

# Recreate activity index from scratch with expected schema.
index_headers = [
    "track_name",
    "activity_url",
    "activity_id",
    "activity_date",
    "activity_type",
    "user_anon_id",
    "file_name",
    "download_status",
    "download_error",
    "downloaded_at",
    "download_attempts",
]

if INDEX_CSV.exists() and INDEX_CSV.stat().st_size > 0:
    backup = INDEX_CSV.with_name(f"activity_index.pre_full_rerun_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv")
    backup.write_bytes(INDEX_CSV.read_bytes())
    print(f"Backed up existing index to: {backup}")

with INDEX_CSV.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=index_headers)
    writer.writeheader()

print(f"Recreated empty index: {INDEX_CSV}")

In [20]:
if AUTO_ACTIVITY_URL_EXPORT:
    banks_peninsula_trail_urls = pd.read_csv("Data/AllTrails/Downloads/banks-peninsula_track_urls.csv")
    for i in range(len(banks_peninsula_trail_urls)):
        
        if banks_peninsula_trail_urls.iloc[i].get("Extracted", 0):
            continue
        else:
            # delay_minutes = random.uniform(MIN_DELAY_MINUTES, MAX_DELAY_MINUTES)
            # print(f"Waiting {delay_minutes:.2f} minutes before next export...")
            # time.sleep(delay_minutes * 60)

            trail_url = banks_peninsula_trail_urls.iloc[i]["track_url"]
            track_name = banks_peninsula_trail_urls.iloc[i]["track_name"]
            print(f"Exporting activity URLs for trail {i+1}/{len(banks_peninsula_trail_urls)}: {track_name}")
            try:
                result = subprocess.run(
                    [
                        sys.executable,
                        "alltrails_activity_urls_export.py",
                        "--url",
                        trail_url,
                        "--out-dir",
                        str(OUT_DIR),
                        "--profile-dir",
                        str(PROFILE_DIR),
                        "--storage-state",
                        str(STORAGE_STATE_FILE),
                        "--index-csv",
                        str(INDEX_CSV),
                        "--id-salt-env",
                        ID_SALT_ENV,
                    ],
                    check=True,
                    cwd=str(Path.cwd()),
                    capture_output=True,
                    text=True,
                )
                if result.stdout:
                    print(result.stdout)
                if result.stderr:
                    print(result.stderr)
                banks_peninsula_trail_urls.loc[i, "Extracted"] = True
            except subprocess.CalledProcessError as exc:
                if exc.stdout:
                    print(exc.stdout)
                if exc.stderr:
                    print(exc.stderr)
                raise
        banks_peninsula_trail_urls.to_csv("Data/AllTrails/Downloads/banks-peninsula_track_urls.csv", index=False)   
            
        
    


Exporting activity URLs for trail 36/37: Mount Herbert from Orton Bradley Park
If login or slider challenge appears, complete it manually in this browser.
Waiting for the activities list to become available...
Still waiting... current URL: https://www.alltrails.com/trail/new-zealand/canterbury/mount-herbert-track-from-orton-bradley-park
Saved session state: C:\Users\maxwe\OneDrive\Documents\Uni_Files\Masters\Code\Data\Alltrails\storage_state.json
Selecting the Activities tab...
Starting Show more exhaustion...
Clicking show more: Show more
Show more click 1: 12/111 URLs found
Last loaded activity year after click 1: 2026
Clicking show more: Show more
Show more click 2: 18/111 URLs found
Last loaded activity year after click 2: 2026
Clicking show more: Show more
Show more click 3: 24/111 URLs found
Last loaded activity year after click 3: 2025
Clicking show more: Show more
Show more click 4: 30/111 URLs found
Last loaded activity year after click 4: 2025
Clicking show more: Show more
Sh

In [16]:
if AUTO_ACTIVITY_EXPORT:
    index_df = pd.read_csv(INDEX_CSV)
    start_row = np.where(index_df['download_status'].isna())[0][0]
    
    cmd = [
        sys.executable,
        "-u",
        "alltrails_activity_data_export.py",
        "--index-csv",
        str(INDEX_CSV),
        "--out-dir",
        str(OUT_DIR),
        "--format",
        "csv",
        "--profile-dir",
        str(PROFILE_DIR),
        "--storage-state",
        str(STORAGE_STATE_FILE),
        "--start-row",
        str(start_row),
        "--max-items",
        "5000",
        "--retries",
        "1",
        "--menu-timeout-ms",
        "10000",
        "--download-timeout-ms",
        "16000",
        "--slow-mo-ms",
        "0",
    ]
    
    # Stream child process logs to notebook output as they are produced.
    process = subprocess.Popen(
        cmd,
        cwd=str(Path.cwd()),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
    
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, cmd)

[83/739] Downloading activity (attempt 1): https://www.alltrails.com/explore/recording/afternoon-hike-at-orton-bradley-waterfall-loop-f6a6765
Saved: C:\Users\maxwe\OneDrive\Documents\Uni_Files\Masters\Code\Data\Alltrails\Downloads\f6a6765.csv
[84/739] Downloading activity (attempt 1): https://www.alltrails.com/explore/recording/afternoon-hike-at-orton-bradley-waterfall-loop-25c4489
Traceback (most recent call last):
  File "c:\Users\maxwe\OneDrive\Documents\Uni_Files\Masters\Code\alltrails_activity_data_export.py", line 388, in <module>
    main()
    ~~~~^^
  File "c:\Users\maxwe\OneDrive\Documents\Uni_Files\Masters\Code\alltrails_activity_data_export.py", line 364, in main
    summary = export_routes_from_index(
        index_csv=index_csv,
    ...<9 lines>...
        slow_mo_ms=max(0, args.slow_mo_ms),
    )
  File "c:\Users\maxwe\OneDrive\Documents\Uni_Files\Masters\Code\alltrails_activity_data_export.py", line 314, in export_routes_from_index
    page.wait_for_timeout(400)
    ~~~

CalledProcessError: Command '['c:\\Users\\maxwe\\anaconda3\\envs\\geospatial_study\\python.exe', '-u', 'alltrails_activity_data_export.py', '--index-csv', 'Data\\AllTrails\\Downloads\\activity_index.csv', '--out-dir', 'Data\\AllTrails\\Downloads', '--format', 'csv', '--profile-dir', 'Data\\AllTrails\\alltrails_profile', '--storage-state', 'Data\\AllTrails\\storage_state.json', '--start-row', '0', '--max-items', '5000', '--retries', '1', '--menu-timeout-ms', '10000', '--download-timeout-ms', '16000', '--slow-mo-ms', '0']' returned non-zero exit status 1.